# 🔧 Notebook 02 — Data Preprocessing
**Project:** ONCOAi — AI-Powered Oral Cancer Detection  
**Author:** Sasmita D (Data Lead) — 727823TUCS305  
**Team:** MediScope | Mentor: Dr. Udhayamoorthi M  

This notebook handles all preprocessing steps — resizing, normalization, augmentation preview, and quality checks — before the data is fed into the model.

## Cell 1 — Mount Drive & Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import random
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Cell 2 — Set Paths & Constants

In [ ]:
# ✅ Update if your folder name is different
DATASET_PATH   = '/content/drive/MyDrive/OncoAi Dataset'
CANCER_PATH    = os.path.join(DATASET_PATH, 'CANCER')
NONCANCER_PATH = os.path.join(DATASET_PATH, 'NON CANCER')

# Target image size for EfficientNetB0
IMG_SIZE = (224, 224)

cancer_files    = [f for f in os.listdir(CANCER_PATH)    if f.lower().endswith(('.jpg','.jpeg','.png'))]
noncancer_files = [f for f in os.listdir(NONCANCER_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))]

print(f'CANCER: {len(cancer_files)} images')
print(f'NON CANCER: {len(noncancer_files)} images')
print(f'Target size: {IMG_SIZE}')

## Cell 3 — Before vs After Resize Comparison

In [ ]:
sample_cancer    = random.choice(cancer_files)
sample_noncancer = random.choice(noncancer_files)

def show_before_after(folder, fname, label, color, axes_row):
    original = Image.open(os.path.join(folder, fname)).convert('RGB')
    resized  = original.resize(IMG_SIZE)

    axes_row[0].imshow(original)
    axes_row[0].set_title(f'{label}\nOriginal: {original.size[0]}x{original.size[1]}px',
                          color=color, fontweight='bold')
    axes_row[0].axis('off')

    axes_row[1].imshow(resized)
    axes_row[1].set_title(f'{label}\nResized: 224x224px',
                          color=color, fontweight='bold')
    axes_row[1].axis('off')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Before vs After Resizing to 224×224', fontsize=15, fontweight='bold')

show_before_after(CANCER_PATH,    sample_cancer,    'CANCER',     'red',   axes[0])
show_before_after(NONCANCER_PATH, sample_noncancer, 'NON CANCER', 'green', axes[1])

plt.tight_layout()
plt.savefig(os.path.join(DATASET_PATH, 'resize_comparison.png'), dpi=150)
plt.show()
print('Resize comparison saved!')

## Cell 4 — Normalization: Before vs After

In [ ]:
img_raw = Image.open(os.path.join(CANCER_PATH, sample_cancer)).convert('RGB').resize(IMG_SIZE)
arr_raw = np.array(img_raw)                    # values: 0 to 255
arr_norm = arr_raw / 255.0                     # values: 0.0 to 1.0

print('--- Normalization Check ---')
print(f'Before normalization → min: {arr_raw.min()}, max: {arr_raw.max()}, dtype: {arr_raw.dtype}')
print(f'After  normalization → min: {arr_norm.min():.3f}, max: {arr_norm.max():.3f}, dtype: {arr_norm.dtype}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Normalization: Pixel Values Before vs After', fontsize=13, fontweight='bold')

axes[0].hist(arr_raw.flatten(), bins=50, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].set_title('Before (0–255)', fontweight='bold')
axes[0].set_xlabel('Pixel Value')
axes[0].set_ylabel('Frequency')

axes[1].hist(arr_norm.flatten(), bins=50, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_title('After  (0.0–1.0)', fontweight='bold')
axes[1].set_xlabel('Pixel Value')

plt.tight_layout()
plt.savefig(os.path.join(DATASET_PATH, 'normalization_comparison.png'), dpi=150)
plt.show()
print('Normalization chart saved!')

## Cell 5 — Data Augmentation Preview

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, array_to_img, load_img

# Define augmentation (same as training pipeline)
aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    shear_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Load one cancer image
img_path = os.path.join(CANCER_PATH, sample_cancer)
img = load_img(img_path, target_size=IMG_SIZE)
x   = img_to_array(img)
x   = x.reshape((1,) + x.shape)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Data Augmentation Preview (CANCER class)', fontsize=14, fontweight='bold')

# Original
axes[0][0].imshow(img)
axes[0][0].set_title('Original', fontweight='bold')
axes[0][0].axis('off')

# 9 augmented versions
gen = aug.flow(x, batch_size=1)
positions = [(0,1),(0,2),(0,3),(0,4),(1,0),(1,1),(1,2),(1,3),(1,4)]
for idx, (r, c) in enumerate(positions):
    aug_img = next(gen)[0]
    axes[r][c].imshow(aug_img)
    axes[r][c].set_title(f'Augmented {idx+1}', fontsize=9)
    axes[r][c].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(DATASET_PATH, 'augmentation_preview.png'), dpi=150)
plt.show()
print('Augmentation preview saved!')

## Cell 6 — Check for Corrupt / Unreadable Images

In [ ]:
def check_corrupt(folder, files, label):
    corrupt = []
    for f in files:
        try:
            img = Image.open(os.path.join(folder, f))
            img.verify()   # checks if file is broken
        except Exception as e:
            corrupt.append((f, str(e)))
    if corrupt:
        print(f'⚠️  {label}: {len(corrupt)} corrupt file(s) found!')
        for fname, err in corrupt:
            print(f'   - {fname}: {err}')
    else:
        print(f'✅ {label}: No corrupt files found. All {len(files)} images are valid.')
    return corrupt

c_corrupt  = check_corrupt(CANCER_PATH,    cancer_files,    'CANCER')
nc_corrupt = check_corrupt(NONCANCER_PATH, noncancer_files, 'NON CANCER')

## Cell 7 — Verify All Images Can Be Resized to 224×224

In [ ]:
def verify_resize(folder, files, label):
    failed = []
    for f in files:
        try:
            img = Image.open(os.path.join(folder, f)).convert('RGB').resize(IMG_SIZE)
            arr = np.array(img)
            assert arr.shape == (224, 224, 3), f'Wrong shape: {arr.shape}'
        except Exception as e:
            failed.append((f, str(e)))
    if failed:
        print(f'⚠️  {label}: {len(failed)} image(s) failed resize check!')
        for fname, err in failed:
            print(f'   - {fname}: {err}')
    else:
        print(f'✅ {label}: All {len(files)} images resize to 224×224 correctly.')

print('Verifying resize compatibility...')
verify_resize(CANCER_PATH,    cancer_files,    'CANCER')
verify_resize(NONCANCER_PATH, noncancer_files, 'NON CANCER')

## Cell 8 — Dataset Split Preview (70 / 15 / 15)

In [ ]:
def split_counts(total, train_r=0.70, val_r=0.15):
    train = int(total * train_r)
    val   = int(total * val_r)
    test  = total - train - val
    return train, val, test

c_train,  c_val,  c_test  = split_counts(len(cancer_files))
nc_train, nc_val, nc_test = split_counts(len(noncancer_files))

print('\n--- Dataset Split Preview (70 / 15 / 15) ---')
print(f'{"Class":<15} {"Train":>8} {"Val":>8} {"Test":>8} {"Total":>8}')
print('-' * 50)
print(f'{"CANCER":<15} {c_train:>8} {c_val:>8} {c_test:>8} {len(cancer_files):>8}')
print(f'{"NON CANCER":<15} {nc_train:>8} {nc_val:>8} {nc_test:>8} {len(noncancer_files):>8}')
print('-' * 50)
print(f'{"TOTAL":<15} {c_train+nc_train:>8} {c_val+nc_val:>8} {c_test+nc_test:>8} {len(cancer_files)+len(noncancer_files):>8}')

# Bar chart of split
fig, ax = plt.subplots(figsize=(9, 5))
splits  = ['Train', 'Validation', 'Test']
cancer_vals    = [c_train,  c_val,  c_test]
noncancer_vals = [nc_train, nc_val, nc_test]
x = np.arange(len(splits))
w = 0.35
ax.bar(x - w/2, cancer_vals,    w, label='CANCER',     color='#e74c3c', edgecolor='black')
ax.bar(x + w/2, noncancer_vals, w, label='NON CANCER', color='#2ecc71', edgecolor='black')
for i, (cv, nv) in enumerate(zip(cancer_vals, noncancer_vals)):
    ax.text(i - w/2, cv + 2, str(cv), ha='center', fontsize=11, fontweight='bold')
    ax.text(i + w/2, nv + 2, str(nv), ha='center', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(splits, fontsize=12)
ax.set_ylabel('Number of Images')
ax.set_title('Dataset Split — Train / Validation / Test', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(DATASET_PATH, 'dataset_split.png'), dpi=150)
plt.show()
print('Split chart saved!')

## Cell 9 — Final Preprocessing Summary

In [ ]:
print('=' * 55)
print('       ONCOAi PREPROCESSING SUMMARY')
print('=' * 55)
print(f'  Input Image Size      : variable (original)')
print(f'  Output Image Size     : 224 × 224 px (RGB)')
print(f'  Normalization         : pixel / 255.0  →  [0.0, 1.0]')
print(f'  Corrupt Images Found  : {len(c_corrupt) + len(nc_corrupt)}')
print(f'  Total Usable Images   : {len(cancer_files) + len(noncancer_files)}')
print('  Augmentation Applied  : rotation, zoom, flip,')
print('                          shear, width/height shift')
print(f'  Train / Val / Test    : 70% / 15% / 15%')
print('=' * 55)
print('\n✅ Preprocessing Complete!')
print('Next step → Run 03_model_training.ipynb (Sedhupathi)')